# SQL
Use joins and parameters to read related data safely.


In [ ]:
# Join related tables and pass values as parameters, never string formatting.
import sqlite3

db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL, owner_id INTEGER NOT NULL);
INSERT INTO users VALUES (1, 'Ada');
INSERT INTO tasks VALUES (1, 'Ship API', 1);
""")
rows = db.execute(
    "SELECT tasks.title, users.name FROM tasks JOIN users ON users.id = tasks.owner_id WHERE users.id = ?",
    (1,),
).fetchall()
print(rows)


## Polished version
Hide SQL behind a small read interface and return typed domain data.


In [ ]:
# Return typed domain data and hide SQL behind a read repository.
from dataclasses import dataclass

@dataclass(frozen=True)
class TaskSummary:
    title: str
    owner: str

class TaskReadRepository:
    def __init__(self, connection: sqlite3.Connection) -> None:
        self.connection = connection

    def for_owner(self, owner_id: int) -> list[TaskSummary]:
        rows = self.connection.execute(
            "SELECT t.title, u.name FROM tasks AS t JOIN users AS u ON u.id = t.owner_id WHERE u.id = ?",
            (owner_id,),
        ).fetchall()
        # Convert database tuples at the repository boundary.
        return [TaskSummary(title, owner) for title, owner in rows]

repository = TaskReadRepository(db)
print(repository.for_owner(1))
